# Курсовая работа: Разработка веб-сервиса для выбора ноутбука методом SIR

**Тема:** Сервис подбора ноутбука для учебы и работы методом SIR (Superiority and Inferiority Ranking).


Разработка веб-сервиса, помогающего пользователям выбрать оптимальную модель ноутбука (или ПК) на основе их бюджета, сферы деятельности и личных предпочтений, используя метод SIR для многокритериальной оценки доступных моделей.

## 1. Разработка системы профиля пользователя и определения критериев выбора

**Задача:** Создать систему профиля пользователя, позволяющую указать цели использования (учеба, программирование, дизайн, игры), бюджет, предпочтения по бренду и форм-фактору, а также определить набор критериев для выбора устройства.

**Функциональные требования:**
*   Регистрация и аутентификация пользователей.
*   Профиль пользователя, содержащий информацию о:
    *   Цели использования (офисные задачи, разработка ПО, монтаж видео, игры).
    *   Бюджете (максимальная сумма покупки).
    *   Предпочитаемом бренде (или отсутствии предпочтений).
    *   Предпочитаемой операционной системе (Windows, macOS, Linux).
    *   Важности различных факторов (вес устройства, время автономной работы, производительность).
*   **Набор критериев выбора ноутбука (7 параметров):**
    1.  $C_1$: Цена (стоимость в рублях) — **минимизация**.
    2.  $C_2$: Производительность процессора (баллы) — **максимизация**.
    3.  $C_3$: Объем RAM (ГБ) — **максимизация**.
    4.  $C_4$: Вес (кг) — **минимизация**.
    5.  $C_5$: Время автономной работы (часы) — **максимизация**.
    6.  $C_6$: Качество экрана (баллы) — **максимизация**.
    7.  $C_7$: Гарантия (месяцы) — **максимизация**.
*   Редактирование профиля и весов кри.

**Технические требования:**
*   Использование базы данных (PostgreSQL) для хранения профилей и весов.
*   Реализация веб-интерфейса для заполнения профиля (использование слайдеров для оценки важности критериев).

## 2. Реализация алгоритма оценки ноутбуков методом SIR

**Задача:** Реализовать алгоритм для оценки и ранжирования доступных моделей ноутбуков на основе многокритериального подхода SIR.

**Функциональные требования:**
*   Обработка данных о каждом ноутбуке по 7 заданным критериям.
*   Расчет оценок моделей с использованием математического аппарата SIR.
*   Формирование итогового списка рекомендаций с учетом индивидуальных весов пользователя.

## 3. Реализация интерфейса для просмотра и сравнения ноутбуков

**Задача:** Разработать интерфейс, позволяющий пользователям просматривать ранжированный список ноутбуков и сравнивать их характеристики.

**Функциональные требования:**
*   Отображение списка ноутбуков, отсортированных по итоговому баллу SIR.
*   Возможность сравнения нескольких ноутбуков по характеристикам.
*   Фильтрация результатов.

**Нефункциональные требования:** 
*   Реализация согласно DDD и событийной архитектуре (см. Персиваль Паттерны на Python).
*   Язык — Python (Реализовать алгоритм для оценки и ранжирования доступных моделей ноутбуков на основе многокритериального подхода SIR); NextJS, TailwindCSS, Zod, ShadCN UI, Framer Motion (для реализация веб-интерфайса).

In [1]:
import pandas as pd
import numpy as np

# --- 1. Входные данные ---
# Создадим DataFrame с 5 тестовыми ноутбуками
data = {
    'Model': ['Lenovo IdeaPad', 'ASUS ZenBook', 'Apple MacBook', 'HP Pavilion', 'Acer Nitro'],
    'C1_Price': [50000, 80000, 120000, 60000, 70000], # Min
    'C2_CPU': [5000, 8000, 10000, 6000, 7500],        # Max
    'C3_RAM': [8, 16, 16, 8, 16],                     # Max
    'C4_Weight': [1.8, 1.2, 1.4, 2.1, 2.3],           # Min
    'C5_Battery': [5, 10, 15, 6, 4],                  # Max
    'C6_Screen': [6, 8, 10, 7, 7],                    # Max
    'C7_Warranty': [12, 24, 12, 12, 24]               # Max
}

df = pd.DataFrame(data)
df.set_index('Model', inplace=True)

# Веса критериев (задаются пользователем, в сумме = 1)
weights = np.array([0.25, 0.20, 0.10, 0.10, 0.15, 0.10, 0.10])

# Направления критериев: 1 = максимизация, -1 = минимизация
directions = np.array([-1, 1, 1, -1, 1, 1, 1])

print("Исходная матрица решений:")
display(df)


# --- 2. Алгоритм SIR ---

# Шаг 1: Нормализация (Min-Max)
norm_df = pd.DataFrame(index=df.index, columns=df.columns)

for i, col in enumerate(df.columns):
    x = df[col].values
    x_min, x_max = x.min(), x.max()
    
    if directions[i] == 1: # Максимизация
        norm_df[col] = (x - x_min) / (x_max - x_min)
    else: # Минимизация
        norm_df[col] = (x_max - x) / (x_max - x_min)

print("\nНормализованная матрица:")
display(norm_df.round(3))

# Шаг 2: Расчет индексов превосходства и неполноценности
alts = norm_df.index
n_alts = len(alts)

superiority = pd.DataFrame(0.0, index=alts, columns=alts)  # π+(a,b)
inferiority = pd.DataFrame(0.0, index=alts, columns=alts)  # π-(a,b)

for a in alts:
    for b in alts:
        if a != b:
            # Считаем разницу по каждому критерию
            diff = norm_df.loc[a] - norm_df.loc[b]
            # Функция предпочтения: учитываем только положительные разницы
            pref_a_over_b = np.maximum(0, diff.values)
            pref_b_over_a = np.maximum(0, -diff.values)
            
            # Умножаем на веса и суммируем
            superiority.loc[a, b] = sum(pref_a_over_b * weights)
            inferiority.loc[a, b] = sum(pref_b_over_a * weights)

# Шаг 3: Глобальные оценки
phi_plus = superiority.sum(axis=1)   # Φ+(a)
phi_minus = inferiority.sum(axis=1)  # Φ-(a)
sir_score = phi_plus - phi_minus     # SIR(a)

# --- 3. Формирование рейтинга ---
results = pd.DataFrame({
    'Superiority (Φ+)': phi_plus,
    'Inferiority (Φ-)': phi_minus,
    'SIR Score': sir_score
})

# Сортируем по убыванию балла SIR
results_sorted = results.sort_values(by='SIR Score', ascending=False)

print("\n--- Итоговый рейтинг ноутбуков (Метод SIR) ---")
display(results_sorted.round(4))

best_laptop = results_sorted.index[0]
print(f"\n🏆 Лучший выбор пользователя: {best_laptop}")

Исходная матрица решений:


,C1_Price,C2_CPU,C3_RAM,C4_Weight,C5_Battery,C6_Screen,C7_Warranty
Model,,,,,,,
Lenovo IdeaPad,50000,5000,8,1.8,5,6,12
ASUS ZenBook,80000,8000,16,1.2,10,8,24
Apple MacBook,120000,10000,16,1.4,15,10,12
HP Pavilion,60000,6000,8,2.1,6,7,12
Acer Nitro,70000,7500,16,2.3,4,7,24



Нормализованная матрица:


,C1_Price,C2_CPU,C3_RAM,C4_Weight,C5_Battery,C6_Screen,C7_Warranty
Model,,,,,,,
Lenovo IdeaPad,1.000,0.0,0.0,0.455,0.091,0.00,0.0
ASUS ZenBook,0.571,0.6,1.0,1.000,0.545,0.50,1.0
Apple MacBook,0.000,1.0,1.0,0.818,1.000,1.00,0.0
HP Pavilion,0.857,0.2,0.0,0.182,0.182,0.25,0.0
Acer Nitro,0.714,0.5,1.0,0.000,0.000,0.25,1.0



--- Итоговый рейтинг ноутбуков (Метод SIR) ---


,Superiority (Φ+),Inferiority (Φ-),SIR Score
Model,,,
ASUS ZenBook,1.4219,0.4125,1.0095
Apple MacBook,1.6991,1.0039,0.6952
Acer Nitro,0.8993,0.8453,0.0540
HP Pavilion,0.4455,1.2857,-0.8402
Lenovo IdeaPad,0.5506,1.4691,-0.9184



🏆 Лучший выбор пользователя: ASUS ZenBook
